# TRACC — Modélisation prédictive des indicateurs climatiques jusqu'en 2100

**Données source** : TRACC (Trajectoires de Réchauffement Adaptées au Changement Climatique)  
**Couverture** : 1976–2005 (REF) + projections GWL15 / GWL20 / GWL30  
**Objectif** : Prédire 15 indicateurs climatiques à l'horizon 2100 selon 3 hypothèses (pessimiste / optimiste / régulière)  
**Algorithmes** : LinearRegression, Ridge, PolynomialRidge, RandomForest, GradientBoosting, HuberRegressor  

---

## Structure du notebook
1. Chargement et préparation des données
2. EDA — Statistiques descriptives et visualisations
3. Modélisation — Comparaison d'algorithmes par indicateur
4. Évaluation — Métriques, diagnostics, comparatif train/test
5. Projection 2006–2100 selon 3 hypothèses
6. Export des résultats au format source

## 1. Imports et configuration

In [ ]:
# Librairies standard scientifiques — pas de dépendance externe au-delà de scikit-learn
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
import scipy.stats as stats
from scipy.stats import pearsonr, spearmanr

# Sklearn — pipeline complet sans dépendance réseau
from sklearn.linear_model import LinearRegression, Ridge, HuberRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, TimeSeriesSplit, learning_curve
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 10
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

SEED = 42
np.random.seed(SEED)

# Palette par hypothèse — convention colorimétrique climatique standard
PALETTE = {
    'optimiste':  '#2196F3',  # bleu : réchauffement contenu
    'reguliere':  '#FF9800',  # orange : trajectoire médiane
    'pessimiste': '#F44336',  # rouge : réchauffement fort
    'historique': '#4CAF50',  # vert : données REF
    'scenarios':  {'REF': '#4CAF50', 'GWL15': '#FFC107', 'GWL20': '#FF9800', 'GWL30': '#F44336'}
}

print(f'NumPy {np.__version__} | Pandas {pd.__version__} | Sklearn importé OK')

ModuleNotFoundError: No module named 'matplotlib'

: 

## 2. Chargement et préparation des données

In [ ]:
# Chargement des trois fichiers TRACC
# tracc_data.json : séries temporelles agrégées (moyenne spatiale)
# tracc_meta.json : labels, catégories, définitions des scénarios
with open('tracc_data.json', 'r') as f:
    raw_data = json.load(f)

with open('tracc_meta.json', 'r') as f:
    meta = json.load(f)

# Construction du DataFrame principal
df_all = pd.DataFrame(raw_data)
df_all['Annee'] = df_all['Annee'].astype(int)
df_all = df_all.sort_values('Annee').reset_index(drop=True)

INDICATEURS = meta['indicateurs']
LABELS = meta['labels']
CATEGORIES = meta['categories']
SCENARIOS_DEF = meta['scenarios']

# Séparation : données historiques (REF 1976-2005) vs projections
df_ref = df_all[df_all['Niveau'] == 'REF'].copy()
df_proj = df_all[df_all['Niveau'] != 'REF'].copy()

print(f'Dataset complet     : {len(df_all)} lignes ({df_all["Annee"].min()}–{df_all["Annee"].max()})')
print(f'Période REF (train) : {len(df_ref)} lignes (1976–2005)')
print(f'Projections (valid) : {len(df_proj)} lignes')
print(f'Niveaux disponibles : {df_all["Niveau"].unique().tolist()}')
print(f'Indicateurs         : {len(INDICATEURS)}')
df_all.head(3)

In [ ]:
# Audit qualité : valeurs manquantes, types, plages
print('=== Audit qualité — Dataset REF ===')
print(df_ref[INDICATEURS].isnull().sum().rename('NaN').to_frame().T)
print()
print('Types de colonnes :', df_ref[INDICATEURS].dtypes.unique())

# Vérification de la continuité temporelle sur REF
annees_ref = sorted(df_ref['Annee'].unique())
gaps = [annees_ref[i+1] - annees_ref[i] for i in range(len(annees_ref)-1)]
print(f'Années REF       : {annees_ref[0]}–{annees_ref[-1]}, n={len(annees_ref)}')
print(f'Pas de temps max : {max(gaps)} (continuité temporelle : {"OK" if max(gaps)==1 else "GAPS detectes"})')

## 3. EDA — Analyse exploratoire

In [ ]:
# Statistiques descriptives complètes sur la période REF
# Ajout du coefficient de variation pour mesurer la dispersion relative
stats_ref = df_ref[INDICATEURS].describe().T
stats_ref['cv_%'] = (stats_ref['std'] / stats_ref['mean'].abs() * 100).round(1)
stats_ref['skewness'] = df_ref[INDICATEURS].skew().round(3)
stats_ref['kurtosis'] = df_ref[INDICATEURS].kurt().round(3)
stats_ref.index = [LABELS[i] for i in stats_ref.index]
print('=== Statistiques descriptives — Période REF (1976–2005) ===')
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_columns', 12)
display(stats_ref[['mean', 'std', 'cv_%', 'min', '25%', '50%', '75%', 'max', 'skewness', 'kurtosis']])

In [ ]:
# Tendances temporelles REF avec régression linéaire et test de Mann-Kendall simplifié
# Choix : Mann-Kendall via scipy.stats.kendalltau (non paramétrique, robuste aux outliers)
print('=== Tendances temporelles REF (1976–2005) ===')
print(f'{"Indicateur":<45} {"Pente (OLS)":>12} {"R²":>8} {"tau Kendall":>12} {"p-value":>10}')
print('-' * 92)
tendances = {}
annees_ref_arr = df_ref['Annee'].values.reshape(-1, 1)
for ind in INDICATEURS:
    y = df_ref[ind].values
    lr = LinearRegression().fit(annees_ref_arr, y)
    y_pred = lr.predict(annees_ref_arr)
    r2 = r2_score(y, y_pred)
    tau, p = stats.kendalltau(df_ref['Annee'].values, y)
    tendances[ind] = {'slope': lr.coef_[0], 'intercept': lr.intercept_, 'r2': r2, 'tau': tau, 'p_kendall': p}
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    print(f'{LABELS[ind]:<45} {lr.coef_[0]:>12.4f} {r2:>8.3f} {tau:>12.4f} {p:>10.4f} {sig}')

In [ ]:
# Visualisation des séries temporelles REF avec tendance OLS
# Grille 5x3 pour les 15 indicateurs
fig, axes = plt.subplots(5, 3, figsize=(18, 20))
axes = axes.flatten()

for i, ind in enumerate(INDICATEURS):
    ax = axes[i]
    y = df_ref[ind].values
    x = df_ref['Annee'].values
    # Série brute
    ax.plot(x, y, color=PALETTE['historique'], alpha=0.7, linewidth=1.2, label='Observé')
    # Tendance OLS
    slope = tendances[ind]['slope']
    intercept = tendances[ind]['intercept']
    ax.plot(x, slope * x + intercept, color='#212121', linewidth=1.5, linestyle='--', label=f'Tendance (pente={slope:.4f}/an)')
    # IC 95% via résidus
    residus = y - (slope * x + intercept)
    ci = 1.96 * residus.std()
    ax.fill_between(x, slope*x+intercept-ci, slope*x+intercept+ci, alpha=0.15, color='#212121')
    ax.set_title(LABELS[ind], fontsize=9, fontweight='bold')
    ax.set_xlabel('Année')
    r2 = tendances[ind]['r2']
    p = tendances[ind]['p_kendall']
    ax.text(0.02, 0.95, f'R²={r2:.3f}  p={p:.3f}', transform=ax.transAxes, fontsize=8, va='top', color='#555')
    if i == 0:
        ax.legend(fontsize=8)

plt.suptitle('Séries temporelles des indicateurs TRACC — Période REF (1976–2005)\nTendance OLS + IC 95%',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_series_temporelles.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Distributions par indicateur + test de normalité Shapiro-Wilk
# Justification : Shapiro-Wilk recommandé pour n<50 (ici n=30)
fig, axes = plt.subplots(5, 3, figsize=(18, 20))
axes = axes.flatten()

for i, ind in enumerate(INDICATEURS):
    ax = axes[i]
    y = df_ref[ind].values
    # Histogramme + KDE
    ax.hist(y, bins=10, color=PALETTE['historique'], alpha=0.6, density=True, edgecolor='white')
    kde_x = np.linspace(y.min(), y.max(), 200)
    kde = stats.gaussian_kde(y)
    ax.plot(kde_x, kde(kde_x), color='#1B5E20', linewidth=2)
    # Ligne médiane
    ax.axvline(np.median(y), color='#F44336', linewidth=1.5, linestyle='--', label=f'Médiane={np.median(y):.2f}')
    # Test Shapiro-Wilk
    stat_sw, p_sw = stats.shapiro(y)
    normal_str = 'Normale' if p_sw > 0.05 else 'Non normale'
    ax.set_title(LABELS[ind], fontsize=9, fontweight='bold')
    ax.text(0.02, 0.95, f'Shapiro p={p_sw:.3f} ({normal_str})\nSkew={stats.skew(y):.2f}',
            transform=ax.transAxes, fontsize=8, va='top', color='#555')
    if i == 0:
        ax.legend(fontsize=8)

plt.suptitle('Distributions des indicateurs TRACC — REF\nHistogramme + KDE + test Shapiro-Wilk',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Matrice de corrélation de Spearman (robuste, non paramétrique)
# Choix Spearman : plusieurs indicateurs sont asymétriques (TX35D, IFM40 fortement skewés)
corr_matrix = df_ref[INDICATEURS].corr(method='spearman')
short_labels = [l.split('(')[0].strip()[:25] for l in [LABELS[i] for i in INDICATEURS]]

fig, ax = plt.subplots(figsize=(14, 12))
mask_upper = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
# Affichage triangle inférieur
im = ax.imshow(np.where(mask_upper, np.nan, corr_matrix.values), cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Corrélation de Spearman')

ax.set_xticks(range(len(INDICATEURS)))
ax.set_yticks(range(len(INDICATEURS)))
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(short_labels, fontsize=8)

# Annotations
for i in range(len(INDICATEURS)):
    for j in range(len(INDICATEURS)):
        if not mask_upper[i, j]:
            val = corr_matrix.iloc[i, j]
            color = 'white' if abs(val) > 0.6 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=6.5, color=color)

ax.set_title('Matrice de corrélation de Spearman — Indicateurs TRACC (REF 1976–2005)', fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('eda_correlation.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Boxplots comparatifs par scénario pour chaque catégorie d'indicateurs
# Montre la shift entre REF -> GWL15 -> GWL20 -> GWL30
niveaux_ordre = ['REF', 'GWL15', 'GWL20', 'GWL30']
couleurs_niv = [PALETTE['scenarios'][n] for n in niveaux_ordre]

for categorie, inds in CATEGORIES.items():
    n = len(inds)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
    if n == 1:
        axes = [axes]
    for ax, ind in zip(axes, inds):
        data_by_niv = [df_all[df_all['Niveau'] == niv][ind].dropna().values for niv in niveaux_ordre]
        bp = ax.boxplot(data_by_niv, patch_artist=True, notch=False, widths=0.5,
                        medianprops={'color': '#212121', 'linewidth': 2})
        for patch, color in zip(bp['boxes'], couleurs_niv):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        ax.set_xticks(range(1, len(niveaux_ordre)+1))
        ax.set_xticklabels(niveaux_ordre)
        ax.set_title(LABELS[ind], fontsize=9, fontweight='bold')
        ax.set_ylabel('Valeur')
        # Annotations moyennes
        for k, vals in enumerate(data_by_niv):
            if len(vals) > 0:
                ax.text(k+1, np.max(vals)*1.02, f'{np.mean(vals):.1f}', ha='center', fontsize=8, color='#333')
    legend_patches = [Patch(facecolor=c, alpha=0.7, label=n) for c, n in zip(couleurs_niv, niveaux_ordre)]
    axes[0].legend(handles=legend_patches, fontsize=8, loc='upper left')
    fig.suptitle(f'Distribution par scénario — Catégorie : {categorie}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'eda_boxplot_{categorie.lower().replace(" ", "_")}.png', bbox_inches='tight', dpi=150)
    plt.show()

In [ ]:
# Q-Q plots pour les indicateurs les plus skewés
# Objectif : valider l'hypothèse de normalité avant modélisation linéaire
inds_skewed = sorted(INDICATEURS, key=lambda i: abs(df_ref[i].skew()), reverse=True)[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, ind in enumerate(inds_skewed):
    ax = axes[i]
    y = df_ref[ind].values
    stats.probplot(y, dist='norm', plot=ax)
    ax.set_title(f'Q-Q plot — {LABELS[ind]}\n(skewness={stats.skew(y):.3f})', fontsize=9, fontweight='bold')
    ax.get_lines()[0].set(color=PALETTE['historique'], markersize=5)
    ax.get_lines()[1].set(color='#F44336', linewidth=1.5)

plt.suptitle('Q-Q plots (normalité) — 6 indicateurs les plus asymétriques', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_qqplots.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. Construction des hypothèses et features

**Hypothèses de projection :**
- **Optimiste** : trajectoire GWL15 (réchauffement contenu à +1.5°C)
- **Régulière** : trajectoire GWL20 (réchauffement médian à +2.0°C)
- **Pessimiste** : trajectoire GWL30 (réchauffement fort à +3.0°C)

Les données REF (1976–2005) servent d'entraînement. Les projections par scénario servent de validation externe.  
Le modèle prédit sur `X = [Annee, Annee², Annee³]` pour capturer les tendances non linéaires.

In [ ]:
# Construction du jeu d'entraînement : données REF uniquement (1976–2005)
# Feature engineering : année centrée pour réduire la colinéarité polynomiale
ANNEE_CENTER = 1990  # centre de la période REF

def build_features(annees, center=ANNEE_CENTER):
    """Retourne un array de features [t, t², t³] avec t = Annee - center."""
    t = np.array(annees) - center
    return np.column_stack([t, t**2, t**3])

X_train = build_features(df_ref['Annee'].values)

# Années de projection continue 2006–2100
annees_futures = np.arange(2006, 2101)
X_future = build_features(annees_futures)

# Années de validation par scénario (pour évaluation out-of-sample)
df_gwl15 = df_all[df_all['Niveau'] == 'GWL15'].sort_values('Annee')
df_gwl20 = df_all[df_all['Niveau'] == 'GWL20'].sort_values('Annee')
df_gwl30 = df_all[df_all['Niveau'] == 'GWL30'].sort_values('Annee')

X_gwl15 = build_features(df_gwl15['Annee'].values)
X_gwl20 = build_features(df_gwl20['Annee'].values)
X_gwl30 = build_features(df_gwl30['Annee'].values)

# Mapping hypothèse -> DataFrame scénario de référence pour recalibration
HYPOTHESES = {
    'optimiste':  {'gwl': 'GWL15', 'df': df_gwl15, 'X': X_gwl15, 'delta': 1.5},
    'reguliere':  {'gwl': 'GWL20', 'df': df_gwl20, 'X': X_gwl20, 'delta': 2.0},
    'pessimiste': {'gwl': 'GWL30', 'df': df_gwl30, 'X': X_gwl30, 'delta': 3.0},
}

print(f'X_train shape : {X_train.shape}')
print(f'X_future shape : {X_future.shape} (2006–2100)')
print(f'Features : [t, t², t³] avec t = Annee - {ANNEE_CENTER}')

## 5. Modélisation — Comparaison d'algorithmes

In [ ]:
# Définition des algorithmes testés
# Justification du choix de chaque algorithme :
#
# LinearRegression    : baseline, interprétable, sensible aux outliers
# Ridge               : régularisation L2, meilleure généralisation, gère la colinéarité t/t²/t³
# PolynomialRidge     : capture les non-linéarités d'ordre 2-3, ridge pour éviter l'overfitting
# RandomForest        : robuste, non paramétrique, capture interactions non linéaires
# GradientBoosting    : boosting séquentiel, excellent sur données tabulaires faibles bruit
# HuberRegressor      : régression robuste aux outliers (utilise perte Huber plutôt que MSE)
# SVR_RBF             : noyau RBF pour capturer des dynamiques non stationnaires

def get_models():
    return {
        'LinearReg': LinearRegression(),
        'Ridge': Ridge(alpha=1.0),
        'PolyRidge': Pipeline([
            ('scaler', StandardScaler()),
            ('poly', PolynomialFeatures(degree=2, include_bias=False)),
            ('ridge', Ridge(alpha=1.0))
        ]),
        'RandomForest': RandomForestRegressor(n_estimators=200, max_depth=5, random_state=SEED),
        'GradientBoosting': GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                                       max_depth=3, random_state=SEED),
        'HuberReg': Pipeline([
            ('scaler', StandardScaler()),
            ('huber', HuberRegressor(epsilon=1.35, max_iter=300))
        ]),
        'SVR_RBF': Pipeline([
            ('scaler', StandardScaler()),
            ('svr', SVR(kernel='rbf', C=10.0, epsilon=0.1))
        ])
    }

print('Algorithmes définis :', list(get_models().keys()))

In [ ]:
# Évaluation croisée avec TimeSeriesSplit (préserve l'ordre temporel)
# Choix TimeSeriesSplit : données temporelles — on ne shuffule pas pour éviter le data leakage
# n_splits=5 sur 30 observations : folds de taille croissante ~5-6 obs

tscv = TimeSeriesSplit(n_splits=5)
MODELES_NOMS = list(get_models().keys())

# Stockage des résultats d'évaluation
resultats_cv = {}  # {indicateur: {modele: {r2, rmse, mae, ...}}}
meilleurs_modeles = {}  # {indicateur: nom_meilleur_modele}

print('=== Évaluation par validation croisée temporelle (TimeSeriesSplit, n_splits=5) ===')
print(f'Train : {X_train.shape[0]} observations | Features : {X_train.shape[1]}')
print()

for ind in INDICATEURS:
    y = df_ref[ind].values
    resultats_cv[ind] = {}
    for nom, model in get_models().items():
        cv_r2 = cross_val_score(model, X_train, y, cv=tscv, scoring='r2')
        cv_neg_rmse = cross_val_score(model, X_train, y, cv=tscv, scoring='neg_root_mean_squared_error')
        cv_neg_mae = cross_val_score(model, X_train, y, cv=tscv, scoring='neg_mean_absolute_error')
        resultats_cv[ind][nom] = {
            'r2_mean': cv_r2.mean(), 'r2_std': cv_r2.std(),
            'rmse_mean': -cv_neg_rmse.mean(), 'rmse_std': cv_neg_rmse.std(),
            'mae_mean': -cv_neg_mae.mean(), 'mae_std': cv_neg_mae.std()
        }
    # Sélection du meilleur modèle : maximisation R² CV
    best = max(resultats_cv[ind], key=lambda m: resultats_cv[ind][m]['r2_mean'])
    meilleurs_modeles[ind] = best

print('Meilleur modèle par indicateur (critère : R² CV moyen) :')
for ind, best in meilleurs_modeles.items():
    r2 = resultats_cv[ind][best]['r2_mean']
    rmse = resultats_cv[ind][best]['rmse_mean']
    print(f'  {LABELS[ind]:<45} -> {best:<20} R²={r2:.3f}  RMSE={rmse:.4f}')

In [ ]:
# Rapport d'évaluation complet : tableau récapitulatif par indicateur × modèle
rows = []
for ind in INDICATEURS:
    for nom in MODELES_NOMS:
        res = resultats_cv[ind][nom]
        rows.append({
            'Indicateur': LABELS[ind],
            'Modele': nom,
            'R2_mean': round(res['r2_mean'], 4),
            'R2_std': round(res['r2_std'], 4),
            'RMSE_mean': round(res['rmse_mean'], 4),
            'RMSE_std': round(res['rmse_std'], 4),
            'MAE_mean': round(res['mae_mean'], 4),
            'Meilleur': nom == meilleurs_modeles[ind]
        })

df_rapport = pd.DataFrame(rows)

def highlight_best(row):
    return ['background-color: #C8E6C9; font-weight: bold' if row['Meilleur'] else '' for _ in row]

print('=== Rapport d\'évaluation complet (TimeSeriesSplit CV) ===')
display(df_rapport.style.apply(highlight_best, axis=1).format({
    'R2_mean': '{:.4f}', 'R2_std': '{:.4f}', 'RMSE_mean': '{:.4f}', 'RMSE_std': '{:.4f}', 'MAE_mean': '{:.4f}'
}))

In [ ]:
# Heatmap R² CV par indicateur × modèle
pivot_r2 = df_rapport.pivot(index='Indicateur', columns='Modele', values='R2_mean')
# Réordonner les modèles
pivot_r2 = pivot_r2[MODELES_NOMS]

fig, ax = plt.subplots(figsize=(14, 9))
im = ax.imshow(pivot_r2.values, cmap='RdYlGn', vmin=-0.5, vmax=1.0, aspect='auto')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='R² (CV TimeSeriesSplit)')

ax.set_xticks(range(len(MODELES_NOMS)))
ax.set_yticks(range(len(pivot_r2)))
ax.set_xticklabels(MODELES_NOMS, rotation=30, ha='right', fontsize=10)
ax.set_yticklabels(pivot_r2.index, fontsize=8)

for i in range(len(pivot_r2)):
    for j in range(len(MODELES_NOMS)):
        val = pivot_r2.values[i, j]
        color = 'white' if val < 0.2 or val > 0.8 else 'black'
        # Marque meilleur modèle
        ind_key = INDICATEURS[i]
        marker = '*' if MODELES_NOMS[j] == meilleurs_modeles[ind_key] else ''
        ax.text(j, i, f'{val:.3f}{marker}', ha='center', va='center', fontsize=8, color=color)

ax.set_title('R² (TimeSeriesSplit CV) par indicateur et algorithme\n(* = meilleur modèle sélectionné)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('eval_heatmap_r2.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Comparatif train / test sur split temporel fixe
# Train : 1976–1998 (23 obs) | Test : 1999–2005 (7 obs)
# Justification du split : respect de l'ordre chronologique, test sur fin de période
SPLIT_YEAR = 1999
mask_train = df_ref['Annee'] < SPLIT_YEAR
mask_test = df_ref['Annee'] >= SPLIT_YEAR

X_tr = build_features(df_ref[mask_train]['Annee'].values)
X_te = build_features(df_ref[mask_test]['Annee'].values)

print(f'Split temporel : Train={mask_train.sum()} obs (1976–{SPLIT_YEAR-1}) | Test={mask_test.sum()} obs ({SPLIT_YEAR}–2005)')
print()

rows_split = []
for ind in INDICATEURS:
    y_tr = df_ref[mask_train][ind].values
    y_te = df_ref[mask_test][ind].values
    for nom, model in get_models().items():
        model.fit(X_tr, y_tr)
        y_pred_tr = model.predict(X_tr)
        y_pred_te = model.predict(X_te)
        rows_split.append({
            'Indicateur': LABELS[ind][:30],
            'Modele': nom,
            'R2_train': round(r2_score(y_tr, y_pred_tr), 4),
            'R2_test': round(r2_score(y_te, y_pred_te), 4),
            'RMSE_train': round(mean_squared_error(y_tr, y_pred_tr)**0.5, 4),
            'RMSE_test': round(mean_squared_error(y_te, y_pred_te)**0.5, 4),
            'MAE_train': round(mean_absolute_error(y_tr, y_pred_tr), 4),
            'MAE_test': round(mean_absolute_error(y_te, y_pred_te), 4),
        })

df_split = pd.DataFrame(rows_split)
df_split['Overfitting_R2'] = (df_split['R2_train'] - df_split['R2_test']).round(4)

print('=== Comparatif Train / Test (split temporel fixe) ===')
display(df_split.sort_values(['Indicateur', 'Modele']))

In [ ]:
# Courbes d'apprentissage pour les meilleurs modèles sur TMm_yr et TX35D_yr
# Permet de diagnostiquer underfitting / overfitting en fonction de la taille d'entraînement
inds_display = ['TMm_yr', 'TX35D_yr']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, ind in zip(axes, inds_display):
    y = df_ref[ind].values
    best_model_name = meilleurs_modeles[ind]
    models_dict = get_models()
    best_model = models_dict[best_model_name]
    train_sizes, train_scores, test_scores = learning_curve(
        best_model, X_train, y,
        cv=TimeSeriesSplit(n_splits=5),
        train_sizes=np.linspace(0.3, 1.0, 8),
        scoring='r2'
    )
    tr_mean = train_scores.mean(axis=1)
    tr_std = train_scores.std(axis=1)
    te_mean = test_scores.mean(axis=1)
    te_std = test_scores.std(axis=1)

    ax.plot(train_sizes, tr_mean, 'o-', color='#2196F3', label='Score train')
    ax.fill_between(train_sizes, tr_mean-tr_std, tr_mean+tr_std, alpha=0.15, color='#2196F3')
    ax.plot(train_sizes, te_mean, 's-', color='#F44336', label='Score validation')
    ax.fill_between(train_sizes, te_mean-te_std, te_mean+te_std, alpha=0.15, color='#F44336')
    ax.axhline(0, color='#999', linewidth=0.8, linestyle=':')
    ax.set_xlabel('Taille du jeu d\'entraînement')
    ax.set_ylabel('R²')
    ax.set_title(f'Courbe d\'apprentissage\n{LABELS[ind]}\n(meilleur modèle : {best_model_name})', fontsize=9, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylim(-1, 1.05)

plt.suptitle('Courbes d\'apprentissage — Diagnostic biais/variance', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('eval_learning_curves.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Diagnostics des résidus sur le meilleur modèle pour les 4 indicateurs principaux
# Checks : normalité, homoscédasticité, autocorrélation (Durbin-Watson)
def durbin_watson(residuals):
    """Statistique DW : 2=pas d'autocorrélation, <2=positive, >2=négative."""
    diff_res = np.diff(residuals)
    return np.sum(diff_res**2) / np.sum(residuals**2)

inds_diag = ['TMm_yr', 'TX35D_yr', 'RR_yr', 'IFM40_yr']
fig, axes = plt.subplots(len(inds_diag), 3, figsize=(16, 4*len(inds_diag)))

for i, ind in enumerate(inds_diag):
    y = df_ref[ind].values
    best_model_name = meilleurs_modeles[ind]
    model = get_models()[best_model_name]
    model.fit(X_train, y)
    y_pred = model.predict(X_train)
    residuals = y - y_pred
    dw = durbin_watson(residuals)
    _, p_sw = stats.shapiro(residuals)

    # Résidus vs fitted
    ax1 = axes[i, 0]
    ax1.scatter(y_pred, residuals, alpha=0.7, color=PALETTE['historique'], edgecolors='white', s=50)
    ax1.axhline(0, color='#F44336', linewidth=1.5, linestyle='--')
    ax1.set_xlabel('Valeurs ajustées')
    ax1.set_ylabel('Résidus')
    ax1.set_title(f'{ind} — Résidus vs Fitted\n(DW={dw:.2f}, Shapiro p={p_sw:.3f})', fontsize=9, fontweight='bold')

    # Q-Q résidus
    ax2 = axes[i, 1]
    stats.probplot(residuals, dist='norm', plot=ax2)
    ax2.set_title(f'{ind} — Q-Q Résidus ({best_model_name})', fontsize=9, fontweight='bold')
    ax2.get_lines()[0].set(color=PALETTE['historique'], markersize=5)
    ax2.get_lines()[1].set(color='#F44336', linewidth=1.5)

    # Résidus temporels (autocorrélation visuelle)
    ax3 = axes[i, 2]
    ax3.plot(df_ref['Annee'].values, residuals, 'o-', color=PALETTE['historique'], markersize=4)
    ax3.axhline(0, color='#F44336', linewidth=1.5, linestyle='--')
    ax3.fill_between(df_ref['Annee'].values, -2*residuals.std(), 2*residuals.std(), alpha=0.1, color='#F44336')
    ax3.set_xlabel('Année')
    ax3.set_ylabel('Résidu')
    ax3.set_title(f'{ind} — Résidus temporels (IC 2σ)', fontsize=9, fontweight='bold')

plt.suptitle('Diagnostics des résidus — 4 indicateurs clés (meilleur modèle)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('eval_residuals.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Projection 2006–2100 selon 3 hypothèses

**Méthode de recalibration par hypothèse :**  
Le meilleur modèle est entraîné sur REF (1976–2005).  
Les projections disponibles (GWL15/20/30) permettent d'estimer un **facteur de décalage** (delta moyen) entre la prédiction du modèle REF et les valeurs observées dans chaque scénario.  
Ce delta est appliqué progressivement sur 2006–2100 pour simuler la trajectoire de réchauffement.  
L'incertitude est modélisée via les résidus du modèle + bruit gaussien contrôlé.

In [ ]:
# Moteur de projection
# Principe : entraîner le meilleur modèle sur REF, calculer le drift par scénario,
#             interpoler le drift sur 2006-2100, ajouter IC via résidus

def compute_scenario_drift(model, X_train, y_train, X_scenario, y_scenario):
    """Calcule le delta moyen entre les valeurs scénario et la prédiction du modèle REF."""
    model.fit(X_train, y_train)
    y_pred_scen = model.predict(X_scenario)
    # Drift = différence moyenne observé_scénario - prédit_par_modèle_REF
    return np.mean(y_scenario - y_pred_scen)

def project_indicator(ind, annees_futures, hypotheses_config, df_ref, verbose=False):
    """
    Projette un indicateur sur annees_futures selon les 3 hypothèses.
    Retourne un dict {hypothese: (annees, y_pred, y_lower, y_upper)}
    """
    y_train = df_ref[ind].values
    best_name = meilleurs_modeles[ind]
    resultats = {}

    for hyp_name, hyp_cfg in hypotheses_config.items():
        model = get_models()[best_name]
        model.fit(X_train, y_train)

        # Calcul du drift entre scénario GWL et prédiction REF
        y_scen = hyp_cfg['df'][ind].values
        drift = compute_scenario_drift(
            get_models()[best_name], X_train, y_train, hyp_cfg['X'], y_scen
        )

        # Projection de base (tendance REF extrapolée)
        y_base = model.predict(X_future)

        # Application progressive du drift : 0% en 2006 -> 100% en 2100
        # Justification : le drift climatique s'accumule progressivement
        progression = (annees_futures - 2006) / (2100 - 2006)
        y_proj = y_base + drift * progression

        # Calcul de l'incertitude via résidus du modèle sur REF
        residuals = y_train - model.predict(X_train)
        sigma = residuals.std()
        # IC 95% : ±1.96σ, élargi progressivement (incertitude croissante dans le temps)
        ic_factor = 1 + 1.5 * progression  # l'incertitude croît avec l'horizon
        y_lower = y_proj - 1.96 * sigma * ic_factor
        y_upper = y_proj + 1.96 * sigma * ic_factor

        resultats[hyp_name] = {
            'annees': annees_futures,
            'y_pred': y_proj,
            'y_lower': y_lower,
            'y_upper': y_upper,
            'drift': drift,
            'modele': best_name
        }
        if verbose:
            print(f'  {hyp_name:<12} drift={drift:+.4f} | modèle={best_name}')

    return resultats

# Lancement de toutes les projections
print('Calcul des projections pour les 15 indicateurs...')
toutes_projections = {}
for ind in INDICATEURS:
    print(f'  {LABELS[ind]}')
    toutes_projections[ind] = project_indicator(ind, annees_futures, HYPOTHESES, df_ref, verbose=True)

print('\nProjections calculées.')

In [ ]:
# Visualisation des projections pour chaque indicateur
# Format : série REF historique + 3 trajectoires + IC 95%
fig, axes = plt.subplots(5, 3, figsize=(20, 25))
axes = axes.flatten()

for i, ind in enumerate(INDICATEURS):
    ax = axes[i]
    proj = toutes_projections[ind]

    # Série historique REF
    ax.plot(df_ref['Annee'], df_ref[ind], color=PALETTE['historique'],
            linewidth=2, label='REF (1976–2005)', zorder=5)

    # Données de validation scénario (points)
    for niv, color in [('GWL15', '#FFC107'), ('GWL20', '#FF9800'), ('GWL30', '#F44336')]:
        df_niv = df_all[df_all['Niveau'] == niv]
        ax.scatter(df_niv['Annee'], df_niv[ind], color=color, s=12, alpha=0.5, zorder=4)

    # Projections par hypothèse
    for hyp_name, color_key in [('optimiste', 'optimiste'), ('reguliere', 'reguliere'), ('pessimiste', 'pessimiste')]:
        p = proj[hyp_name]
        color = PALETTE[color_key]
        ax.plot(p['annees'], p['y_pred'], color=color, linewidth=1.8,
                label=f'{hyp_name.capitalize()} ({p["modele"]})')
        ax.fill_between(p['annees'], p['y_lower'], p['y_upper'], color=color, alpha=0.12)

    ax.axvline(2005, color='#555', linewidth=1, linestyle=':', alpha=0.7)
    ax.set_title(LABELS[ind], fontsize=9, fontweight='bold')
    ax.set_xlabel('Année', fontsize=8)

    if i == 0:
        ax.legend(fontsize=7, loc='upper left')

plt.suptitle('Projections climatiques 2006–2100 — 3 hypothèses\nIC 95% (bandes) | Points = données GWL de validation',
             fontsize=14, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig('projections_2100.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Projections détaillées par catégorie (meilleure lisibilité)
for categorie, inds in CATEGORIES.items():
    n = len(inds)
    fig, axes = plt.subplots(1, n, figsize=(7*n, 6))
    if n == 1:
        axes = [axes]
    for ax, ind in zip(axes, inds):
        proj = toutes_projections[ind]
        # REF historique
        ax.plot(df_ref['Annee'], df_ref[ind], color=PALETTE['historique'],
                linewidth=2.5, label='REF observé', zorder=5)
        # Données GWL (validation)
        for niv, niv_color in PALETTE['scenarios'].items():
            if niv == 'REF':
                continue
            df_niv = df_all[df_all['Niveau'] == niv]
            ax.scatter(df_niv['Annee'], df_niv[ind], color=niv_color, s=18, alpha=0.6,
                       label=f'{niv} observé', zorder=4)
        # Hypothèses
        for hyp_name in ['optimiste', 'reguliere', 'pessimiste']:
            p = proj[hyp_name]
            color = PALETTE[hyp_name]
            ax.plot(p['annees'], p['y_pred'], color=color, linewidth=2,
                    label=f'Hyp. {hyp_name} (drift={p["drift"]:+.3f})', zorder=5)
            ax.fill_between(p['annees'], p['y_lower'], p['y_upper'], color=color, alpha=0.1)

        ax.axvline(2005, color='#555', linewidth=1.2, linestyle=':', label='Fin REF')
        ax.set_title(LABELS[ind], fontsize=10, fontweight='bold')
        ax.set_xlabel('Année')
        ax.legend(fontsize=8)

    fig.suptitle(f'Projections détaillées — {categorie}\n(avec drift de recalibration par scénario)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'proj_{categorie.lower().replace(" ", "_")}.png', bbox_inches='tight', dpi=150)
    plt.show()

In [ ]:
# Validation externe : comparaison des projections avec les données GWL disponibles
# Métriques calculées sur les points GWL vs prédiction du modèle REF + drift
print('=== Validation externe — Prédiction vs données GWL observées ===')
print(f'{"Indicateur":<40} {"Hyp.":<12} {"R²":>8} {"RMSE":>10} {"MAE":>10} {"Biais":>10}')
print('-' * 94)

validation_rows = []
for ind in INDICATEURS:
    for hyp_name, hyp_cfg in HYPOTHESES.items():
        # Prédiction sur les années GWL du scénario correspondant
        best_name = meilleurs_modeles[ind]
        model = get_models()[best_name]
        y_train_ind = df_ref[ind].values
        model.fit(X_train, y_train_ind)

        annees_scen = hyp_cfg['df']['Annee'].values
        X_scen_feat = build_features(annees_scen)
        y_obs = hyp_cfg['df'][ind].values

        drift = toutes_projections[ind][hyp_name]['drift']
        progression = (annees_scen - 2006) / (2100 - 2006)
        progression = np.clip(progression, 0, 1)
        y_pred_scen = model.predict(X_scen_feat) + drift * progression

        r2_v = r2_score(y_obs, y_pred_scen)
        rmse_v = mean_squared_error(y_obs, y_pred_scen)**0.5
        mae_v = mean_absolute_error(y_obs, y_pred_scen)
        biais = np.mean(y_pred_scen - y_obs)

        validation_rows.append({
            'Indicateur': LABELS[ind][:38],
            'Hypothese': hyp_name,
            'R2': round(r2_v, 4),
            'RMSE': round(rmse_v, 4),
            'MAE': round(mae_v, 4),
            'Biais': round(biais, 4)
        })
        print(f'{LABELS[ind][:38]:<40} {hyp_name:<12} {r2_v:>8.3f} {rmse_v:>10.4f} {mae_v:>10.4f} {biais:>+10.4f}')

df_validation = pd.DataFrame(validation_rows)

In [ ]:
# Heatmap de validation externe R² par indicateur x hypothèse
pivot_val = df_validation.pivot(index='Indicateur', columns='Hypothese', values='R2')
pivot_val = pivot_val[['optimiste', 'reguliere', 'pessimiste']]

fig, ax = plt.subplots(figsize=(8, 10))
im = ax.imshow(pivot_val.values, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='R² — Validation externe (données GWL)')
ax.set_xticks(range(3))
ax.set_yticks(range(len(pivot_val)))
ax.set_xticklabels(['Optimiste', 'Reguliere', 'Pessimiste'], fontsize=11)
ax.set_yticklabels(pivot_val.index, fontsize=8)

for i in range(len(pivot_val)):
    for j in range(3):
        val = pivot_val.values[i, j]
        color = 'white' if abs(val) > 0.6 else 'black'
        ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=9, color=color)

ax.set_title('R² Validation externe — Prédictions vs données GWL\n(indicateurs × hypothèses)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('eval_validation_externe.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Tableau synthétique : valeurs projetées à 2050 et 2100 par indicateur × hypothèse
# Format comparable à la structure source TRACC
rows_synth = []
for ind in INDICATEURS:
    proj = toutes_projections[ind]
    for hyp_name in ['optimiste', 'reguliere', 'pessimiste']:
        p = proj[hyp_name]
        idx_2050 = np.where(p['annees'] == 2050)[0][0]
        idx_2100 = np.where(p['annees'] == 2100)[0][0]
        val_ref_mean = df_ref[ind].mean()
        val_2050 = p['y_pred'][idx_2050]
        val_2100 = p['y_pred'][idx_2100]
        rows_synth.append({
            'Indicateur': ind,
            'Label': LABELS[ind],
            'Hypothese': hyp_name,
            'Modele': p['modele'],
            'REF_mean': round(val_ref_mean, 4),
            'Proj_2050': round(val_2050, 4),
            'Proj_2100': round(val_2100, 4),
            'Delta_2050': round(val_2050 - val_ref_mean, 4),
            'Delta_2100': round(val_2100 - val_ref_mean, 4),
            'Drift': round(proj[hyp_name]['drift'], 4)
        })

df_synth = pd.DataFrame(rows_synth)
print('=== Synthèse des projections 2050 et 2100 ===')
display(df_synth.style.background_gradient(
    subset=['Delta_2050', 'Delta_2100'],
    cmap='RdBu_r', vmin=-5, vmax=5
).format({'REF_mean': '{:.3f}', 'Proj_2050': '{:.3f}', 'Proj_2100': '{:.3f}',
          'Delta_2050': '{:+.3f}', 'Delta_2100': '{:+.3f}', 'Drift': '{:+.4f}'}))

In [ ]:
# Visualisation des deltas à 2100 par indicateur × hypothèse
fig, ax = plt.subplots(figsize=(16, 7))
x = np.arange(len(INDICATEURS))
width = 0.25

for k, hyp_name in enumerate(['optimiste', 'reguliere', 'pessimiste']):
    deltas = [df_synth[(df_synth['Indicateur']==ind) & (df_synth['Hypothese']==hyp_name)]['Delta_2100'].values[0]
              for ind in INDICATEURS]
    bars = ax.bar(x + k*width, deltas, width, label=hyp_name.capitalize(),
                  color=PALETTE[hyp_name], alpha=0.8, edgecolor='white')

ax.axhline(0, color='#212121', linewidth=1, linestyle='--')
ax.set_xticks(x + width)
ax.set_xticklabels([LABELS[i].split('(')[0][:22] for i in INDICATEURS], rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Delta vs REF (même unité que l\'indicateur)')
ax.set_title('Anomalie projetée à 2100 vs moyenne REF — par indicateur et hypothèse',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('proj_delta_2100.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Export des résultats au format TRACC source

In [ ]:
# Export des projections dans le même format JSON que tracc_data.json
# Chaque hypothèse génère une série d'entrées annuelles 2006–2100
# Structure identique : {Annee, Niveau, indicateurs...}

NIVEAU_MAP = {
    'optimiste': 'PROJ_OPT',
    'reguliere': 'PROJ_REG',
    'pessimiste': 'PROJ_PES'
}

output_records = []
for hyp_name, niveau_code in NIVEAU_MAP.items():
    for idx_a, annee in enumerate(annees_futures):
        record = {
            'Annee': int(annee),
            'Niveau': niveau_code,
            'Hypothese': hyp_name,
            'nb_points': None  # non applicable sur projection
        }
        for ind in INDICATEURS:
            p = toutes_projections[ind][hyp_name]
            record[ind] = round(float(p['y_pred'][idx_a]), 4)
            record[f'{ind}_lower95'] = round(float(p['y_lower'][idx_a]), 4)
            record[f'{ind}_upper95'] = round(float(p['y_upper'][idx_a]), 4)
        output_records.append(record)

with open('tracc_projections_2100.json', 'w', encoding='utf-8') as f:
    json.dump(output_records, f, ensure_ascii=False, indent=2)

print(f'Export JSON : {len(output_records)} enregistrements (3 hypothèses × 95 années)')
print(f'Fichier : tracc_projections_2100.json')
print(f'Colonnes par enregistrement : Annee, Niveau, Hypothese + {len(INDICATEURS)} indicateurs × 3 (pred, lower, upper)')
pd.DataFrame(output_records[:3])

In [ ]:
# Export CSV pour usage BI (Power BI, Tableau, etc.)
# Format long : une ligne par (Annee, Hypothese, Indicateur)
rows_long = []
# Données historiques REF d'abord
for _, row in df_ref.iterrows():
    for ind in INDICATEURS:
        rows_long.append({
            'Annee': row['Annee'],
            'Source': 'historique',
            'Niveau': 'REF',
            'Hypothese': 'reference',
            'Indicateur': ind,
            'Label': LABELS[ind],
            'Categorie': next(c for c, ids in CATEGORIES.items() if ind in ids),
            'Valeur': round(row[ind], 4),
            'Valeur_lower95': None,
            'Valeur_upper95': None
        })

# Projections
for hyp_name, niveau_code in NIVEAU_MAP.items():
    for idx_a, annee in enumerate(annees_futures):
        for ind in INDICATEURS:
            p = toutes_projections[ind][hyp_name]
            rows_long.append({
                'Annee': int(annee),
                'Source': 'projection',
                'Niveau': niveau_code,
                'Hypothese': hyp_name,
                'Indicateur': ind,
                'Label': LABELS[ind],
                'Categorie': next(c for c, ids in CATEGORIES.items() if ind in ids),
                'Valeur': round(float(p['y_pred'][idx_a]), 4),
                'Valeur_lower95': round(float(p['y_lower'][idx_a]), 4),
                'Valeur_upper95': round(float(p['y_upper'][idx_a]), 4)
            })

df_export_long = pd.DataFrame(rows_long)
df_export_long.to_csv('tracc_projections_long.csv', index=False, encoding='utf-8')
print(f'Export CSV long : {len(df_export_long)} lignes')
print('Colonnes :', df_export_long.columns.tolist())
df_export_long.head(5)

In [ ]:
# Export du rapport d'évaluation et de validation
df_rapport.to_csv('rapport_evaluation_cv.csv', index=False)
df_validation.to_csv('rapport_validation_externe.csv', index=False)
df_synth.to_csv('synthese_projections.csv', index=False)

print('Fichiers exportés :')
print('  tracc_projections_2100.json  — projections au format source TRACC + IC95%')
print('  tracc_projections_long.csv   — format long BI (historique + projections)')
print('  rapport_evaluation_cv.csv    — R², RMSE, MAE par modèle × indicateur (CV)')
print('  rapport_validation_externe.csv — validation out-of-sample sur données GWL')
print('  synthese_projections.csv     — deltas 2050/2100 vs REF')

## 8. Synthèse décisionnelle

### Choix techniques retenus

1. **Features polynomiales [t, t², t³]** centrées sur 1990 — capture la non-linéarité des tendances climatiques, réduit la colinéarité par centrage.
2. **TimeSeriesSplit (5 folds)** — validation croisée sans data leakage temporel, adapté aux séries continues.
3. **7 algorithmes testés** — de la régression linéaire au GradientBoosting, couvrant le spectre biais/variance. Le meilleur est sélectionné par R² CV.
4. **Drift de recalibration** — le delta entre scénario GWL et prédiction REF est appliqué progressivement sur 2006-2100, ancrant les projections sur les données climatiques réelles.
5. **IC 95% croissant** — l'incertitude est modélisée via les résidus du modèle, amplifiée avec l'horizon temporel (facteur 1 à 2.5).
6. **3 hypothèses** — Optimiste=GWL15 (+1.5°C), Régulière=GWL20 (+2°C), Pessimiste=GWL30 (+3°C).
7. **Double export** — JSON (format source TRACC) + CSV long (format BI pivot-ready).

### Limites
- 30 observations REF : taille limitée pour les modèles non paramétriques (RF, GB moins fiables).
- Saut temporel 2005→2037 : interpolation sans données entre les deux périodes.
- Drift calculé sur la moyenne GWL : ne capture pas la variabilité intra-scénario.
- Pas de modélisation des ruptures (ex. 2003 canicule) ni des effets de seuil.